In [ ]:
import cv2
import os
import pandas as pd 

In [ ]:
face_cascade=cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
face_cascade

In [ ]:
cam = cv2.VideoCapture(0)
while True :
    ret,frame=cam.read()
    if not ret:
        print("camera open nhi ho rha hai")
        break
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)   
    faces=face_cascade.detectMultiScale(gray,scaleFactor=1.3,minNeighbors=5)
    for x,y,w,h in faces:
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.imshow("Face Detection",frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cam.release() 
cv2.destroyAllWindows()
        
    

In [ ]:
dataset_path="dataset"
students_path="students"
os.makedirs(dataset_path,exist_ok=True)
os.makedirs(students_path,exist_ok=True)
name=input("Enter student name : ").strip()
existing_ids=[]
for folder in os.listdir(dataset_path):
    if folder.isdigit():
         existing_ids.append(int(folder))
        
if existing_ids:
    student_id=max(existing_ids) + 1
else:
    student_id = 1

print("Student ID:", student_id)
print("Student Name:", name)   

In [ ]:
students_file = os.path.join(students_path, "students.csv")

new_student = pd.DataFrame({
    "ID": [student_id],
    "Name": [name]
})

if os.path.exists(students_file):
    students_df = pd.read_csv(students_file)
    students_df = pd.concat([students_df, new_student], ignore_index=True)
else:
    students_df = new_student

students_df.to_csv(students_file, index=False)

print("Student information saved successfully!")

In [ ]:
import cv2
import os

# Student ke ID ka folder
student_folder = os.path.join(dataset_path, str(student_id))
os.makedirs(student_folder, exist_ok=True)

# Face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

camera = cv2.VideoCapture(0)

count = 0

print("Camera start ho raha hai...")
print("Face ko camera ke saamne rakho.")

while True:

    ret, frame = camera.read()

    if not ret:
        print("Camera open nahi ho raha.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        # Face ke around box
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (255, 0, 0),
            2
        )

        # Face image crop
        face = gray[y:y+h, x:x+w]

        # Image save
        count += 1

        image_path = os.path.join(
            student_folder,
            f"{count}.jpg"
        )

        cv2.imwrite(image_path, face)

        # Screen par count show
        cv2.putText(
            frame,
            f"Images: {count}/30",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    cv2.imshow("Face Capture", frame)

    # 30 images complete
    if count >= 30:
        break

    # q press karke manually stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()

print(f"Capture complete! Total images saved: {count}")                    

In [ ]:
import os
import cv2
import numpy as np

X = []
y = []

for student_id_folder in os.listdir(dataset_path):

    # Sirf numeric folders ko student ID maanenge
    if not student_id_folder.isdigit():
        continue

    student_id_value = int(student_id_folder)
    student_folder = os.path.join(dataset_path, student_id_folder)

    for image_name in os.listdir(student_folder):

        image_path = os.path.join(student_folder, image_name)

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            continue

        # CNN ke liye same image size
        image = cv2.resize(image, (100, 100))

        X.append(image)
        y.append(student_id_value)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("Total images:", len(X))
print("Image shape:", X.shape)
print("Labels:", y)
print("Unique students:", np.unique(y))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Student IDs ko CNN ke class numbers mein convert karna
label_encoder = LabelEncoder()
encoded = label_encoder.fit_transform(y)

# Image mein channel dimension add karna
X = X.reshape(-1, 100, 100, 1)

# Pixel values 0-255 → 0-1
X = X / 255.0

# Training aur testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    encoded,
    test_size=0.2,
    random_state=42,
    stratify= encoded
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))
print("Classes:", label_encoder.classes_)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(100, 100, 1)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(len(label_encoder.classes_), activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=8,
    validation_data=(X_test, y_test)
)

In [ ]:

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy * 100, "%")

In [ ]:
import os

trainer_path = "trainer"
os.makedirs(trainer_path, exist_ok=True)

print("Trainer folder ready!")
model_path = os.path.join(trainer_path, "face_cnn.keras")

model.save(model_path)

print("CNN model saved successfully!")
print("Model path:", model_path)

In [ ]:
import pandas as pd

label_map = pd.DataFrame({
    "CNN_Class": range(len(label_encoder.classes_)),
    "Student_ID": label_encoder.classes_
})

label_map_path = os.path.join(trainer_path, "label_map.csv")

label_map.to_csv(label_map_path, index=False)

print("Label mapping saved successfully!")
print(label_map)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# Saved CNN model load karo
model = tf.keras.models.load_model(model_path)

# Face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

camera = cv2.VideoCapture(0)

while True:

    ret, frame = camera.read()

    if not ret:
        print("Camera open nahi ho raha.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        # Face crop
        face = gray[y:y+h, x:x+w]

        # CNN ke same size mein resize
        face = cv2.resize(face, (100, 100))

        # Normalize
        face = face.astype("float32") / 255.0

        # CNN input shape
        face = face.reshape(1, 100, 100, 1)

        # Prediction
        prediction = model.predict(face, verbose=0)

        predicted_class = np.argmax(prediction[0])
        confidence = np.max(prediction[0])

        # CNN class → actual Student ID
        student_id_value = label_encoder.classes_[predicted_class]

        # Box
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (255, 0, 0),
            2
        )

        # Result
        text = f"ID: {student_id_value}  Confidence: {confidence*100:.1f}%"

        cv2.putText(
            frame,
            text,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow("CNN Face Recognition", frame)

    # q press karke stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()

In [ ]:
import os
import pandas as pd
from datetime import datetime

# Attendance folder
attendance_path = "attendance"

# Folder nahi hai to create ho jayega
os.makedirs(attendance_path, exist_ok=True)

# Attendance CSV ka path
attendance_file = os.path.join(
    attendance_path,
    "attendance.csv"
)

# Agar CSV nahi hai to create karo
if not os.path.exists(attendance_file):

    attendance_df = pd.DataFrame(
        columns=["ID", "Name", "Date", "Time", "Status"]
    )

    attendance_df.to_csv(
        attendance_file,
        index=False
    )

print("Attendance file ready!")
print("File:", attendance_file)

In [ ]:
def mark_attendance(student_id_value):

    # Students ki information read karo
    students_df = pd.read_csv(students_file)

    # Student ID se name find karo
    student_row = students_df[
        students_df["ID"] == student_id_value
    ]

    if student_row.empty:
        print("Student information not found.")
        return

    student_name = student_row.iloc[0]["Name"]

    # Current date aur time
    now = datetime.now()

    today = now.strftime("%Y-%m-%d")
    current_time = now.strftime("%H:%M:%S")

    # Existing attendance read karo
    attendance_df = pd.read_csv(attendance_file)

    # Check karo ki aaj attendance already marked hai ya nahi
    already_marked = attendance_df[
        (attendance_df["ID"] == student_id_value) &
        (attendance_df["Date"] == today)
    ]

    if not already_marked.empty:

        print(
            f"Attendance already marked for {student_name} today."
        )

    else:

        new_attendance = pd.DataFrame({
            "ID": [student_id_value],
            "Name": [student_name],
            "Date": [today],
            "Time": [current_time],
            "Status": ["Present"]
        })

        attendance_df = pd.concat(
            [attendance_df, new_attendance],
            ignore_index=True
        )

        attendance_df.to_csv(
            attendance_file,
            index=False
        )

        print(
            f"Attendance marked for {student_name}!"
        )

In [ ]:
import cv2
import numpy as np

camera = cv2.VideoCapture(0)

while True:

    ret, frame = camera.read()

    if not ret:
        print("Camera open nahi ho raha.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        # Face crop
        face = gray[y:y+h, x:x+w]

        # CNN ke required size mein
        face = cv2.resize(face, (100, 100))

        # Normalize
        face = face.astype("float32") / 255.0

        # CNN input shape
        face = face.reshape(1, 100, 100, 1)

        # Prediction
        prediction = model.predict(face, verbose=0)

        predicted_class = np.argmax(prediction[0])
        confidence = np.max(prediction[0])

        # Confidence threshold
        if confidence >= 0.70:

            student_id_value = int(
                label_encoder.classes_[predicted_class]
            )

            # Attendance mark
            mark_attendance(student_id_value)

            # Screen par ID
            text = f"ID: {student_id_value}  {confidence*100:.1f}%"

        else:

            text = "Unknown Face"

        # Face box
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (255, 0, 0),
            2
        )

        cv2.putText(
            frame,
            text,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow("Face Attendance System", frame)

    # q press karke camera close
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()